# ViT-Motion — ALIGNED interpretability of the intervention

Fixes the mismatch from the previous run: we now **penalize, measure, and visualize the
SAME quantity** — input-gradient saliency of the `yaw` target — so the before/after is
consistent with what the fine-tune actually optimized.

Attach: code + data + artifacts. GPU + Internet ON.


In [ ]:
# setup (robust): stage code, deps, locate data/checkpoint, build/repair manifest
import os, sys, glob, shutil, pathlib
print('Attached inputs:'); [print('   ',d) for d in sorted(glob.glob('/kaggle/input/*'))]
codes=glob.glob('/kaggle/input/**/inspect_dataset.py',recursive=True)
assert codes,'CODE dataset not attached.'; CODE_SRC=str(pathlib.Path(sorted(codes,key=len)[0]).parent)
WORK='/kaggle/working/vit_motion_project'
if os.path.exists(WORK): shutil.rmtree(WORK)
shutil.copytree(CODE_SRC,WORK); os.chdir(WORK); sys.path.insert(0,WORK)
get_ipython().system("pip -q install 'timm>=1.0' 'opencv-python>=4.9' >/dev/null")
csvs=[c for c in glob.glob('/kaggle/input/**/samples.csv',recursive=True) if not c.startswith(CODE_SRC)]
assert csvs,'DATA dataset not attached (no samples.csv).'
DATA_ROOT=sorted({str(pathlib.Path(c).parent.parent) for c in csvs},key=len)[0]
cks=glob.glob('/kaggle/input/**/best.pt',recursive=True); assert cks,'ARTIFACTS (best.pt) not attached.'
CKPT_SRC=sorted(cks,key=len)[0]; print('DATA',DATA_ROOT); print('CKPT',CKPT_SRC)
import yaml; yaml.safe_dump(yaml.safe_load(open('config_kaggle.yaml')),open('config_run.yaml','w'),sort_keys=False,allow_unicode=True)
os.makedirs('artifacts/manifest',exist_ok=True)
import pandas as pd; man=glob.glob('/kaggle/input/**/manifest/manifest.csv',recursive=True); reuse=False
if man:
    mm=pd.read_csv(man[0])
    if 'rgb_abs_path' in mm and os.path.exists(str(mm['rgb_abs_path'].iloc[0])):
        src=str(pathlib.Path(man[0]).parent)
        for f in ['manifest.csv','normalization.json','splits.json']:
            if os.path.exists(os.path.join(src,f)): shutil.copy(os.path.join(src,f),'artifacts/manifest/'+f)
        reuse=True; print('reused prebuilt manifest')
if not reuse:
    print('rebuilding manifest (~5-6 min)...'); get_ipython().system('python inspect_dataset.py --config config_run.yaml --data-root "{DATA_ROOT}"')
CKPT='artifacts/runs/vit_motion_temporal_cr_v0_2_1/best.pt'; os.makedirs(os.path.dirname(CKPT),exist_ok=True); shutil.copy(CKPT_SRC,CKPT)
m=pd.read_csv('artifacts/manifest/manifest.csv'); pool=m[m.split=='test'] if (m.split=='test').any() else m
EXP=sorted(pool.experiment_id.astype(str).unique())[0]; print('experiment',EXP)

In [ ]:
import base64, os, pathlib
# SYNC LATEST CODE: overwrite the 3 updated files into the working project,
# so --saliency / input_saliency work regardless of the code dataset version.
WORK='/kaggle/working/vit_motion_project'
_FILES = {
  'vit_motion/interpret.py': '...',
  'interpret_quantify.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJRdWFudGlmeSBXSEVSRSB0aGUgdmlzdWFsIGVuY29kZXIgbG9va3M6IHNreSAoYWJvdmUgaG9yaXpvbikgdnMgZ3JvdW5kLgoKVHVybnMgdGhlIHF1YWxpdGF0aXZlIEdyYWQtQ0FNIG9ic2VydmF0aW9uICgidGhlIG1vZGVsIHNlZW1zIHRvIGxvb2sgYXQgdGhlIHNreSIpCmludG8gYSBudW1iZXI6IHRoZSBmcmFjdGlvbiBvZiBHcmFkLUNBTSBzYWxpZW5jeSBtYXNzIHRoYXQgZmFsbHMgaW4gdGhlIHNreSByZWdpb24KdnMuIHRoZSBncm91bmQgcmVnaW9uLCBhZ2dyZWdhdGVkIG92ZXIgbWFueSB3aW5kb3dzIGFuZCBleHBlcmltZW50cy4gQmVjYXVzZSB0aGUKc2t5IG9jY3VwaWVzIGEga25vd24gZnJhY3Rpb24gb2YgdGhlIGZyYW1lLCB3ZSBjYW4gYWxzbyByZXBvcnQgYSBzYWxpZW5jeS10by1hcmVhCnJhdGlvOiA+MSBtZWFucyB0aGUgbW9kZWwgb3Zlci1hdHRlbmRzIHRvIHRoZSBza3kgcmVsYXRpdmUgdG8gaXRzIGFyZWEuCgpObyByZXRyYWluaW5nIG5lZWRlZCDigJQgcnVucyBvbiBhbiBleGlzdGluZyBjaGVja3BvaW50LgoKRXhhbXBsZToKICAgIHB5dGhvbiBpbnRlcnByZXRfcXVhbnRpZnkucHkgLS1jb25maWcgY29uZmlnX3J1bi55YW1sIFwKICAgICAgICAtLWNoZWNrcG9pbnQgYXJ0aWZhY3RzL3J1bnMvdml0X21vdGlvbl90ZW1wb3JhbF9jcl92MF8yXzEvYmVzdC5wdCBcCiAgICAgICAgLS1leHBlcmltZW50cyBhdXRvIC0tbnVtLXBlci1leHAgNDAgLS10YXJnZXQgeWF3IC0taG9yaXpvbi1mcmFjIDAuNQoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG1hdHBsb3RsaWIKbWF0cGxvdGxpYi51c2UoIkFnZyIpCmltcG9ydCBtYXRwbG90bGliLnB5cGxvdCBhcyBwbHQKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IHRvcmNoCgpmcm9tIHZpdF9tb3Rpb24uY29uZmlnIGltcG9ydCBsb2FkX2NvbmZpZywgcmVzb2x2ZV9mcm9tX2NvbmZpZwpmcm9tIHZpdF9tb3Rpb24udmFsaWRhdGlvbiBpbXBvcnQgRXhwZXJpbWVudERhdGFzZXQsIGxvYWRfbW9kZWwsIGxvYWRfbm9ybWFsaXphdGlvbgpmcm9tIHZpdF9tb3Rpb24uaW50ZXJwcmV0IGltcG9ydCBNb3Rpb25JbnRlcnByZXRlcgoKCmRlZiBza3lfZnJhY3Rpb24oY2FtOiBucC5uZGFycmF5LCBob3Jpem9uX2ZyYWM6IGZsb2F0KSAtPiBmbG9hdDoKICAgICIiIkZyYWN0aW9uIG9mIEdyYWQtQ0FNIG1hc3MgYWJvdmUgdGhlIGhvcml6b24gcm93ICh0aGUgc2t5IHJlZ2lvbikuIiIiCiAgICBoID0gbWF4KDEsIGludChyb3VuZChjYW0uc2hhcGVbMF0gKiBob3Jpem9uX2ZyYWMpKSkKICAgIHRvdGFsID0gZmxvYXQoY2FtLnN1bSgpKQogICAgaWYgdG90YWwgPD0gMDoKICAgICAgICByZXR1cm4gZmxvYXQoIm5hbiIpCiAgICByZXR1cm4gZmxvYXQoY2FtWzpoXS5zdW0oKSkgLyB0b3RhbAoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIHAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcC5hZGRfYXJndW1lbnQoIi0tY29uZmlnIiwgZGVmYXVsdD0iY29uZmlnLnlhbWwiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tY2hlY2twb2ludCIsIHJlcXVpcmVkPVRydWUpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1leHBlcmltZW50cyIsIGRlZmF1bHQ9ImF1dG8iLAogICAgICAgICAgICAgICAgICAgaGVscD0iJ2F1dG8nID0gZmlyc3QgLS1tYXgtZXhwIHRlc3QgZXhwZXJpbWVudHMsIG9yIGEgY29tbWEgbGlzdCBvZiBpZHMuIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLW1heC1leHAiLCB0eXBlPWludCwgZGVmYXVsdD02KQogICAgcC5hZGRfYXJndW1lbnQoIi0tbnVtLXBlci1leHAiLCB0eXBlPWludCwgZGVmYXVsdD00MCkKICAgIHAuYWRkX2FyZ3VtZW50KCItLXRhcmdldCIsIGRlZmF1bHQ9InlhdyIsIGNob2ljZXM9WyJkeCIsICJkeSIsICJ5YXciLCAibm9ybSJdKQogICAgcC5hZGRfYXJndW1lbnQoIi0tc2FsaWVuY3kiLCBkZWZhdWx0PSJncmFkY2FtIiwgY2hvaWNlcz1bImdyYWRjYW0iLCAiaW5wdXRncmFkIl0sCiAgICAgICAgICAgICAgICAgICBoZWxwPSJpbnB1dGdyYWQgPSBhbGlnbmVkIHdpdGggdGhlIFJSUiBwZW5hbHR5OyBncmFkY2FtID0gdGhlIHZpc3VhbCBtYXAuIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLXNtb290aC1zaWdtYSIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9My4wKQogICAgcC5hZGRfYXJndW1lbnQoIi0taG9yaXpvbi1mcmFjIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjUsCiAgICAgICAgICAgICAgICAgICBoZWxwPSJUb3AgZnJhY3Rpb24gb2YgdGhlIGZyYW1lIHRyZWF0ZWQgYXMgc2t5ICgwLjUgPSB0b3AgaGFsZikuIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLW91dHB1dC1kaXIiLCBkZWZhdWx0PSJhcnRpZmFjdHMvcXVhbnRpZnkiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tdGFnIiwgZGVmYXVsdD0iYmFzZWxpbmUiLCBoZWxwPSJMYWJlbCBmb3Igb3V0cHV0cyAoZS5nLiBiYXNlbGluZSAvIGZpbmV0dW5lZCkuIikKICAgIGFyZ3MgPSBwLnBhcnNlX2FyZ3MoKQoKICAgIGNmZyA9IGxvYWRfY29uZmlnKGFyZ3MuY29uZmlnKQogICAgY29uZmlnX3BhdGggPSBjZmdbIl9jb25maWdfcGF0aCJdCiAgICBtYW5pZmVzdF9kaXIgPSByZXNvbHZlX2Zyb21fY29uZmlnKGNmZ1siZGF0YSJdWyJtYW5pZmVzdF9kaXIiXSwgY29uZmlnX3BhdGgpCiAgICBvdXRfZGlyID0gcmVzb2x2ZV9mcm9tX2NvbmZpZyhhcmdzLm91dHB1dF9kaXIsIGNvbmZpZ19wYXRoKQogICAgb3V0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgbWFuaWZlc3QgPSBwZC5yZWFkX2NzdihtYW5pZmVzdF9kaXIgLyAibWFuaWZlc3QuY3N2IikKICAgIGlmIGFyZ3MuZXhwZXJpbWVudHMgPT0gImF1dG8iOgogICAgICAgIHBvb2wgPSBtYW5pZmVzdFttYW5pZmVzdFsic3BsaXQiXSA9PSAidGVzdCJdIGlmIChtYW5pZmVzdFsic3BsaXQiXSA9PSAidGVzdCIpLmFueSgpIGVsc2UgbWFuaWZlc3QKICAgICAgICBleHBzID0gc29ydGVkKHBvb2xbImV4cGVyaW1lbnRfaWQiXS5hc3R5cGUoc3RyKS51bmlxdWUoKSlbOiBhcmdzLm1heF9leHBdCiAgICBlbHNlOgogICAgICAgIGV4cHMgPSBbZS5zdHJpcCgpIGZvciBlIGluIGFyZ3MuZXhwZXJpbWVudHMuc3BsaXQoIiwiKSBpZiBlLnN0cmlwKCldCgogICAgbm9ybWFsaXphdGlvbiA9IGxvYWRfbm9ybWFsaXphdGlvbihtYW5pZmVzdF9kaXIgLyAibm9ybWFsaXphdGlvbi5qc29uIikKICAgIGRldmljZSA9IHRvcmNoLmRldmljZSgiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKQogICAgbW9kZWwgPSBsb2FkX21vZGVsKGFyZ3MuY2hlY2twb2ludCwgY2ZnLCBkZXZpY2UpCiAgICBpbnRlcnAgPSBNb3Rpb25JbnRlcnByZXRlcihtb2RlbCwgZGV2aWNlKQogICAgaW1hZ2Vfc2l6ZSA9IHR1cGxlKGludCh4KSBmb3IgeCBpbiBjZmdbImRhdGEiXVsiaW1hZ2Vfc2l6ZSJdKQogICAgc2FtcGxlX3BlcmlvZCA9IGZsb2F0KGNmZ1siZGF0YSJdLmdldCgic2FtcGxlX3BlcmlvZF9zZWMiLCAwLjEpKQoKICAgIGZyYWN0aW9uczogbGlzdFtmbG9hdF0gPSBbXQogICAgcGVyX2V4cDogZGljdFtzdHIsIGZsb2F0XSA9IHt9CiAgICBmb3IgZWlkIGluIGV4cHM6CiAgICAgICAgZnJhbWUgPSBtYW5pZmVzdFttYW5pZmVzdFsiZXhwZXJpbWVudF9pZCJdLmFzdHlwZShzdHIpID09IGVpZF0uY29weSgpCiAgICAgICAgaWYgZnJhbWUuZW1wdHk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgZHMgPSBFeHBlcmltZW50RGF0YXNldChmcmFtZSwgbm9ybWFsaXphdGlvbiwgaW1hZ2Vfc2l6ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1vZGVsLnNlcXVlbmNlX2xlbmd0aCwgbW9kZWwuaW1hZ2VfdXBkYXRlX2ludGVydmFsLCBzYW1wbGVfcGVyaW9kKQogICAgICAgIG4gPSBtaW4oYXJncy5udW1fcGVyX2V4cCwgbGVuKGRzKSkKICAgICAgICBpZHhzID0gbnAubGluc3BhY2UoMCwgbGVuKGRzKSAtIDEsIG4sIGR0eXBlPWludCkKICAgICAgICB2YWxzID0gW10KICAgICAgICBmb3IgaSBpbiBpZHhzOgogICAgICAgICAgICBpdGVtID0gZHNbaW50KGkpXQogICAgICAgICAgICBjYW0sIF8gPSBpbnRlcnAuc2FsaWVuY3lfbWFwKGl0ZW1bImltYWdlIl0sIGl0ZW1bIm51bWVyaWMiXSwgaXRlbVsiaW1hZ2VfYWdlIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdGFyZ2V0PWFyZ3MudGFyZ2V0LCBtZXRob2Q9YXJncy5zYWxpZW5jeSwgc21vb3RoX3NpZ21hPWFyZ3Muc21vb3RoX3NpZ21hKQogICAgICAgICAgICBmID0gc2t5X2ZyYWN0aW9uKGNhbSwgYXJncy5ob3Jpem9uX2ZyYWMpCiAgICAgICAgICAgIGlmIG5vdCBucC5pc25hbihmKToKICAgICAgICAgICAgICAgIGZyYWN0aW9ucy5hcHBlbmQoZikKICAgICAgICAgICAgICAgIHZhbHMuYXBwZW5kKGYpCiAgICAgICAgaWYgdmFsczoKICAgICAgICAgICAgcGVyX2V4cFtlaWRdID0gZmxvYXQobnAubWVhbih2YWxzKSkKICAgICAgICBwcmludChmIiAge2VpZH06IHNreS1mcmFjdGlvbiB7bnAubWVhbih2YWxzKTouM2Z9ICAobj17bGVuKHZhbHMpfSkiLCBmbHVzaD1UcnVlKQoKICAgIGZyYWN0aW9ucyA9IG5wLmFzYXJyYXkoZnJhY3Rpb25zLCBkdHlwZT1mbG9hdCkKICAgIHNreV9hcmVhID0gYXJncy5ob3Jpem9uX2ZyYWMKICAgIHN1bW1hcnkgPSB7CiAgICAgICAgInRhZyI6IGFyZ3MudGFnLAogICAgICAgICJ0YXJnZXQiOiBhcmdzLnRhcmdldCwKICAgICAgICAic2FsaWVuY3kiOiBhcmdzLnNhbGllbmN5LAogICAgICAgICJob3Jpem9uX2ZyYWMiOiBhcmdzLmhvcml6b25fZnJhYywKICAgICAgICAiZXhwZXJpbWVudHMiOiBleHBzLAogICAgICAgICJuX3NhbXBsZXMiOiBpbnQoZnJhY3Rpb25zLnNpemUpLAogICAgICAgICJza3lfc2FsaWVuY3lfZnJhY3Rpb25fbWVhbiI6IGZsb2F0KGZyYWN0aW9ucy5tZWFuKCkpIGlmIGZyYWN0aW9ucy5zaXplIGVsc2UgTm9uZSwKICAgICAgICAic2t5X3NhbGllbmN5X2ZyYWN0aW9uX3N0ZCI6IGZsb2F0KGZyYWN0aW9ucy5zdGQoKSkgaWYgZnJhY3Rpb25zLnNpemUgZWxzZSBOb25lLAogICAgICAgICJncm91bmRfc2FsaWVuY3lfZnJhY3Rpb25fbWVhbiI6IGZsb2F0KDEgLSBmcmFjdGlvbnMubWVhbigpKSBpZiBmcmFjdGlvbnMuc2l6ZSBlbHNlIE5vbmUsCiAgICAgICAgInNreV9hcmVhX2ZyYWN0aW9uIjogc2t5X2FyZWEsCiAgICAgICAgInNreV9zYWxpZW5jeV90b19hcmVhX3JhdGlvIjogZmxvYXQoZnJhY3Rpb25zLm1lYW4oKSAvIHNreV9hcmVhKSBpZiBmcmFjdGlvbnMuc2l6ZSBlbHNlIE5vbmUsCiAgICAgICAgInBlcl9leHBlcmltZW50X21lYW4iOiBwZXJfZXhwLAogICAgfQogICAgKG91dF9kaXIgLyBmInNreV9ncm91bmRfe2FyZ3MudGFnfS5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKHN1bW1hcnksIGluZGVudD0yKSkKCiAgICAjIEZpZ3VyZTogZGlzdHJpYnV0aW9uIG9mIHBlci1zYW1wbGUgc2t5LWZyYWN0aW9uIHdpdGggYXJlYSBiYXNlbGluZSArIG1lYW4uCiAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KDguNSwgNSksIGNvbnN0cmFpbmVkX2xheW91dD1UcnVlKQogICAgYXguaGlzdChmcmFjdGlvbnMsIGJpbnM9MjQsIHJhbmdlPSgwLCAxKSwgY29sb3I9IiMxNTY1QzAiLCBhbHBoYT0wLjg1LCBlZGdlY29sb3I9IndoaXRlIikKICAgIG0gPSBmbG9hdChmcmFjdGlvbnMubWVhbigpKSBpZiBmcmFjdGlvbnMuc2l6ZSBlbHNlIDAuMAogICAgYXguYXh2bGluZShza3lfYXJlYSwgY29sb3I9IiM1NTUiLCBscz0iLS0iLCBsdz0xLjUsCiAgICAgICAgICAgICAgIGxhYmVsPWYic2t5IGFyZWEgc2hhcmUgPSB7c2t5X2FyZWE6LjAlfSAodW5pZm9ybSBiYXNlbGluZSkiKQogICAgYXguYXh2bGluZShtLCBjb2xvcj0iI0VGNkMwMCIsIGx3PTIuNSwgbGFiZWw9ZiJtZWFuIHNreS1zYWxpZW5jeSA9IHttOi4xJX0iKQogICAgYXguc2V0X3hsYWJlbCgiZnJhY3Rpb24gb2YgR3JhZC1DQU0gbWFzcyBpbiB0aGUgU0tZIHJlZ2lvbiAocGVyIHNhbXBsZSkiKQogICAgYXguc2V0X3lsYWJlbCgibnVtYmVyIG9mIHNhbXBsZXMiKQogICAgcmF0aW8gPSBtIC8gc2t5X2FyZWEgaWYgc2t5X2FyZWEgZWxzZSBmbG9hdCgibmFuIikKICAgIGF4LnNldF90aXRsZSgKICAgICAgICBmIldoZXJlIGRvZXMgdGhlIHZpc3VhbCBlbmNvZGVyIGxvb2s/ICAoe2FyZ3MudGFnfSwgdGFyZ2V0PXthcmdzLnRhcmdldH0pXG4iCiAgICAgICAgZiJza3kgZ2V0cyB7bTouMCV9IG9mIHNhbGllbmN5IGJ1dCBpcyB7c2t5X2FyZWE6LjAlfSBvZiB0aGUgZnJhbWUgIgogICAgICAgIGYi4oaSIHNhbGllbmN5LXRvLWFyZWEgcmF0aW8ge3JhdGlvOi4yZn3DlyIKICAgICkKICAgIGF4LmxlZ2VuZCgpCiAgICBmaWcuc2F2ZWZpZyhvdXRfZGlyIC8gZiJza3lfZ3JvdW5kX3thcmdzLnRhZ30ucG5nIiwgZHBpPTE3MCkKICAgIHBsdC5jbG9zZShmaWcpCgogICAgcHJpbnQoIlxuPT09IFNLWSB2cyBHUk9VTkQgc2FsaWVuY3kgPT09IikKICAgIHByaW50KGpzb24uZHVtcHMoc3VtbWFyeSwgaW5kZW50PTIpKQogICAgcHJpbnQoZiJcblNhdmVkIHRvOiB7b3V0X2Rpcn0iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK',
  'compare_interpret.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJCZWZvcmUvYWZ0ZXIgR3JhZC1DQU06IHNob3cgYXR0ZW50aW9uIG1vdmluZyBmcm9tIHNreSB0byBncm91bmQgYWZ0ZXIgZmluZS10dW5pbmcuCgpGb3IgdGhlIHNhbWUgd2luZG93cywgcmVuZGVyIGEgZ3JpZDogZWFjaCByb3cgaXMgb25lIHNhbXBsZSwgY29sdW1ucyBhcmUKW2NhY2hlZCBSR0IsIGJhc2VsaW5lIEdyYWQtQ0FNLCBmaW5lLXR1bmVkIEdyYWQtQ0FNXSwgd2l0aCB0aGUgcGVyLXNhbXBsZSBza3kKc2FsaWVuY3kgZnJhY3Rpb24gYW5ub3RhdGVkIHNvIHRoZSBzaGlmdCBpcyB2aXNpYmxlIEFORCBxdWFudGlmaWVkLgoKRXhhbXBsZToKICAgIHB5dGhvbiBjb21wYXJlX2ludGVycHJldC5weSAtLWNvbmZpZyBjb25maWdfcnVuLnlhbWwgXAogICAgICAgIC0tY2hlY2twb2ludC1iYXNlbGluZSBhcnRpZmFjdHMvcnVucy92aXRfbW90aW9uX3RlbXBvcmFsX2NyX3YwXzJfMS9iZXN0LnB0IFwKICAgICAgICAtLWNoZWNrcG9pbnQtZmluZXR1bmVkIGFydGlmYWN0cy9ydW5zL3ZpdF9tb3Rpb25fcnJyL2Jlc3RfcnJyLnB0IFwKICAgICAgICAtLWV4cGVyaW1lbnQgMjAyNjA4MDRfbW9ybmluZ19kcnlfMDA0IC0tbnVtLXNhbXBsZXMgNCAtLXRhcmdldCB5YXcKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBtYXRwbG90bGliCm1hdHBsb3RsaWIudXNlKCJBZ2ciKQppbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0CmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKCmZyb20gdml0X21vdGlvbi5jb25maWcgaW1wb3J0IGxvYWRfY29uZmlnLCByZXNvbHZlX2Zyb21fY29uZmlnCmZyb20gdml0X21vdGlvbi52YWxpZGF0aW9uIGltcG9ydCBFeHBlcmltZW50RGF0YXNldCwgbG9hZF9leHBlcmltZW50X2ZyYW1lLCBsb2FkX21vZGVsLCBsb2FkX25vcm1hbGl6YXRpb24KZnJvbSB2aXRfbW90aW9uLmludGVycHJldCBpbXBvcnQgTW90aW9uSW50ZXJwcmV0ZXIsIGRlbm9ybWFsaXplX2ltYWdlCgoKZGVmIHNreV9mcmFjKGNhbSwgaG9yaXpvbl9mcmFjPTAuNSk6CiAgICBoID0gbWF4KDEsIGludChyb3VuZChjYW0uc2hhcGVbMF0gKiBob3Jpem9uX2ZyYWMpKSkKICAgIHRvdCA9IGZsb2F0KGNhbS5zdW0oKSkKICAgIHJldHVybiBmbG9hdChjYW1bOmhdLnN1bSgpKSAvIHRvdCBpZiB0b3QgPiAwIGVsc2UgZmxvYXQoIm5hbiIpCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWNvbmZpZyIsIGRlZmF1bHQ9ImNvbmZpZy55YW1sIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1jaGVja3BvaW50LWJhc2VsaW5lIiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1jaGVja3BvaW50LWZpbmV0dW5lZCIsIHJlcXVpcmVkPVRydWUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZXhwZXJpbWVudCIsIHJlcXVpcmVkPVRydWUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbnVtLXNhbXBsZXMiLCB0eXBlPWludCwgZGVmYXVsdD00KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXRhcmdldCIsIGRlZmF1bHQ9InlhdyIsIGNob2ljZXM9WyJkeCIsICJkeSIsICJ5YXciLCAibm9ybSJdKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNhbGllbmN5IiwgZGVmYXVsdD0iaW5wdXRncmFkIiwgY2hvaWNlcz1bImdyYWRjYW0iLCAiaW5wdXRncmFkIl0sCiAgICAgICAgICAgICAgICAgICAgaGVscD0iaW5wdXRncmFkID0gdGhlIFNBTUUgcXVhbnRpdHkgZmluZXR1bmVfcnJyIHBlbmFsaXplcyAoYWxpZ25lZCwgcmVjb21tZW5kZWQpLiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc21vb3RoLXNpZ21hIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0zLjApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0taG9yaXpvbi1mcmFjIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0cHV0IiwgZGVmYXVsdD0iYXJ0aWZhY3RzL2ludGVycHJldGFiaWxpdHkvYmVmb3JlX2FmdGVyLnBuZyIpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgY2ZnID0gbG9hZF9jb25maWcoYXJncy5jb25maWcpCiAgICBjcCA9IGNmZ1siX2NvbmZpZ19wYXRoIl0KICAgIG1hbmlmZXN0X2RpciA9IHJlc29sdmVfZnJvbV9jb25maWcoY2ZnWyJkYXRhIl1bIm1hbmlmZXN0X2RpciJdLCBjcCkKICAgIG91dF9wYXRoID0gcmVzb2x2ZV9mcm9tX2NvbmZpZyhhcmdzLm91dHB1dCwgY3ApCiAgICBvdXRfcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKICAgIGZyYW1lLCBlaWQgPSBsb2FkX2V4cGVyaW1lbnRfZnJhbWUobWFuaWZlc3RfZGlyIC8gIm1hbmlmZXN0LmNzdiIsIGFyZ3MuZXhwZXJpbWVudCkKICAgIG5vcm0gPSBsb2FkX25vcm1hbGl6YXRpb24obWFuaWZlc3RfZGlyIC8gIm5vcm1hbGl6YXRpb24uanNvbiIpCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKICAgIG1fYmFzZSA9IGxvYWRfbW9kZWwoYXJncy5jaGVja3BvaW50X2Jhc2VsaW5lLCBjZmcsIGRldmljZSkKICAgIG1fZnQgPSBsb2FkX21vZGVsKGFyZ3MuY2hlY2twb2ludF9maW5ldHVuZWQsIGNmZywgZGV2aWNlKQogICAgaW1hZ2Vfc2l6ZSA9IHR1cGxlKGludCh4KSBmb3IgeCBpbiBjZmdbImRhdGEiXVsiaW1hZ2Vfc2l6ZSJdKQogICAgc3AgPSBmbG9hdChjZmdbImRhdGEiXS5nZXQoInNhbXBsZV9wZXJpb2Rfc2VjIiwgMC4xKSkKICAgIGRzID0gRXhwZXJpbWVudERhdGFzZXQoZnJhbWUsIG5vcm0sIGltYWdlX3NpemUsIG1fYmFzZS5zZXF1ZW5jZV9sZW5ndGgsIG1fYmFzZS5pbWFnZV91cGRhdGVfaW50ZXJ2YWwsIHNwKQoKICAgIGliID0gTW90aW9uSW50ZXJwcmV0ZXIobV9iYXNlLCBkZXZpY2UpCiAgICBpZm4gPSBNb3Rpb25JbnRlcnByZXRlcihtX2Z0LCBkZXZpY2UpCiAgICBuID0gbWluKGFyZ3MubnVtX3NhbXBsZXMsIGxlbihkcykpCiAgICBpZHhzID0gbnAubGluc3BhY2UoMCwgbGVuKGRzKSAtIDEsIG4sIGR0eXBlPWludCkKCiAgICBmaWcsIGF4ZXMgPSBwbHQuc3VicGxvdHMobiwgMywgZmlnc2l6ZT0oMTEsIDMuNCAqIG4pLCBjb25zdHJhaW5lZF9sYXlvdXQ9VHJ1ZSkKICAgIGlmIG4gPT0gMToKICAgICAgICBheGVzID0gYXhlc1tOb25lLCA6XQogICAgZm9yIHIsIGkgaW4gZW51bWVyYXRlKGlkeHMpOgogICAgICAgIGl0ZW0gPSBkc1tpbnQoaSldCiAgICAgICAgcmdiID0gZGVub3JtYWxpemVfaW1hZ2UoaXRlbVsiaW1hZ2UiXSkKICAgICAgICBjYW1fYiwgXyA9IGliLnNhbGllbmN5X21hcChpdGVtWyJpbWFnZSJdLCBpdGVtWyJudW1lcmljIl0sIGl0ZW1bImltYWdlX2FnZSJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRhcmdldD1hcmdzLnRhcmdldCwgbWV0aG9kPWFyZ3Muc2FsaWVuY3ksIHNtb290aF9zaWdtYT1hcmdzLnNtb290aF9zaWdtYSkKICAgICAgICBjYW1fZiwgXyA9IGlmbi5zYWxpZW5jeV9tYXAoaXRlbVsiaW1hZ2UiXSwgaXRlbVsibnVtZXJpYyJdLCBpdGVtWyJpbWFnZV9hZ2UiXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdGFyZ2V0PWFyZ3MudGFyZ2V0LCBtZXRob2Q9YXJncy5zYWxpZW5jeSwgc21vb3RoX3NpZ21hPWFyZ3Muc21vb3RoX3NpZ21hKQogICAgICAgIGZiLCBmZiA9IHNreV9mcmFjKGNhbV9iLCBhcmdzLmhvcml6b25fZnJhYyksIHNreV9mcmFjKGNhbV9mLCBhcmdzLmhvcml6b25fZnJhYykKICAgICAgICBheGVzW3IsIDBdLmltc2hvdyhyZ2IpOyBheGVzW3IsIDBdLmF4aXMoIm9mZiIpCiAgICAgICAgaWYgciA9PSAwOgogICAgICAgICAgICBheGVzW3IsIDBdLnNldF90aXRsZSgiQ2FjaGVkIFJHQiIsIGZvbnRzaXplPTEzKQogICAgICAgIGF4ZXNbciwgMV0uaW1zaG93KHJnYik7IGF4ZXNbciwgMV0uaW1zaG93KGNhbV9iLCBjbWFwPSJqZXQiLCBhbHBoYT0wLjQ1KTsgYXhlc1tyLCAxXS5heGlzKCJvZmYiKQogICAgICAgIGF4ZXNbciwgMV0uc2V0X3RpdGxlKGYiQmFzZWxpbmUg4oCUIHNreSB7ZmI6LjAlfSIgKyAoIiIgaWYgciBlbHNlICIgIChHcmFkLUNBTSkiKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb250c2l6ZT0xMiwgY29sb3I9IiNCMzI2MUUiKQogICAgICAgIGF4ZXNbciwgMl0uaW1zaG93KHJnYik7IGF4ZXNbciwgMl0uaW1zaG93KGNhbV9mLCBjbWFwPSJqZXQiLCBhbHBoYT0wLjQ1KTsgYXhlc1tyLCAyXS5heGlzKCJvZmYiKQogICAgICAgIGF4ZXNbciwgMl0uc2V0X3RpdGxlKGYiRmluZS10dW5lZCDigJQgc2t5IHtmZjouMCV9IiArICgiIiBpZiByIGVsc2UgIiAgKEdyYWQtQ0FNKSIpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvbnRzaXplPTEyLCBjb2xvcj0iIzFCN0EzRCIpCiAgICBmaWcuc3VwdGl0bGUoZiJXaGVyZSB0aGUgbW9kZWwgbG9va3MsIGJlZm9yZSB2cy4gYWZ0ZXIgYXR0ZW50aW9uLWd1aWRlZCBmaW5lLXR1bmluZyAiCiAgICAgICAgICAgICAgICAgZiIoe2VpZH0sIHRhcmdldD17YXJncy50YXJnZXR9LCBzYWxpZW5jeT17YXJncy5zYWxpZW5jeX0pIiwgZm9udHNpemU9MTMpCiAgICBmaWcuc2F2ZWZpZyhvdXRfcGF0aCwgZHBpPTE2MCkKICAgIHBsdC5jbG9zZShmaWcpCiAgICBwcmludChmIlNhdmVkIGJlZm9yZS9hZnRlciBHcmFkLUNBTSB0bzoge291dF9wYXRofSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=',
}
for rel,b in _FILES.items():
    p=os.path.join(WORK,rel); os.makedirs(os.path.dirname(p),exist_ok=True)
    pathlib.Path(p).write_bytes(base64.b64decode(b)); print('synced',rel)

## 1) Fine-tune with the ALIGNED target (yaw), penalizing input-grad sky saliency


In [ ]:
RRR='artifacts/runs/vit_motion_rrr_yaw/best_rrr_yaw.pt'
!python finetune_rrr.py --config config_run.yaml --checkpoint "{CKPT}" \
  --epochs 3 --lambda-rrr 1.0 --horizon-frac 0.5 --target yaw --max-steps 200 --output "{RRR}"

## 2) Aligned quantification: input-grad sky fraction, baseline vs fine-tuned
Same quantity as the penalty → the number and the picture agree.


In [ ]:
!python interpret_quantify.py --config config_run.yaml --checkpoint "{CKPT}" --experiments auto --max-exp 6 --num-per-exp 40 --target yaw --saliency inputgrad --tag base_ig
!python interpret_quantify.py --config config_run.yaml --checkpoint "{RRR}" --experiments auto --max-exp 6 --num-per-exp 40 --target yaw --saliency inputgrad --tag ft_ig
import json
b=json.load(open('artifacts/quantify/sky_ground_base_ig.json')); f=json.load(open('artifacts/quantify/sky_ground_ft_ig.json'))
print('ALIGNED sky saliency (input-grad, yaw):')
print('  baseline  mean %.3f  std %.3f'%(b['sky_saliency_fraction_mean'],b['sky_saliency_fraction_std']))
print('  finetuned mean %.3f  std %.3f'%(f['sky_saliency_fraction_mean'],f['sky_saliency_fraction_std']))
from IPython.display import Image, display
display(Image('artifacts/quantify/sky_ground_base_ig.png')); display(Image('artifacts/quantify/sky_ground_ft_ig.png'))

## 3) Aligned before/after maps (input-grad, yaw) — should now be consistent


In [ ]:
!python compare_interpret.py --config config_run.yaml \
  --checkpoint-baseline "{CKPT}" --checkpoint-finetuned "{RRR}" \
  --experiment "{EXP}" --num-samples 4 --target yaw --saliency inputgrad --smooth-sigma 3.0 \
  --output artifacts/interpretability/before_after_aligned.png
from IPython.display import Image, display; display(Image('artifacts/interpretability/before_after_aligned.png'))

## 4) Accuracy check of the aligned fine-tuned model


In [ ]:
!python evaluate_experiment.py --config config_run.yaml --checkpoint "{RRR}" --experiment "{EXP}" --output-dir artifacts/evaluation_rrr_yaw
import json; e=json.load(open(f'artifacts/evaluation_rrr_yaw/{EXP}/metrics.json'))['outputs']
for k in ['next_body_dx','next_delta_yaw']: print(k,'R2=%.3f MAE=%.5f'%(e[k]['r2'],e[k]['mae']))

In [ ]:
import shutil; shutil.make_archive('/kaggle/working/vit_motion_aligned','zip','artifacts'); print('done -> vit_motion_aligned.zip')